# Compare Vabamorf with Muna homonyms dataset


## Table of contents

1. [**Gathering Data**](#andmete_kogumine)
2. [**Evaluation**](#hindamine)
3. [**Results**](#tulemused)

[end](#end)


In [1]:
import os
import json
import warnings
import types
import pandas as pd
import numpy as np
import estnltk, estnltk.converters, estnltk.taggers
import sklearn

# from bert_morph_tagger_notebook_functions import NotebookFunctions

# from simpletransformers.ner import NERModel, NERArgs
from tqdm import tqdm
from bert_morph_tagger import BertMorphTagger
from estnltk.converters.label_studio.labelling_configurations import (
    PhraseClassificationConfiguration,
)
from estnltk.converters.label_studio.labelling_tasks import PhraseClassificationTask

e:\Git_projects\EstNLTK\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<a id='andmete_kogumine'></a>


### Gathering data


In [30]:
# Use specific annotation configurations that were used in homonyms dataset
annotation_confs = {
    1: PhraseClassificationConfiguration(
        phrase_labels=["analüüsitav sõna"],
        class_labels={"sg n": "sg n", "sg g": "sg g"},
        header="Vali sõna morfoloogiline vorm (sg n - ainsuse nimetav, sg g -- ainsuse omastav):",
        header_placement="middle",
    ),
    16: PhraseClassificationConfiguration(
        phrase_labels=["analüüsitav sõna"],
        class_labels={"sg n": "sg n", "sg g": "sg g"},
        header="Vali sõna morfoloogiline vorm (sg n - ainsuse nimetav, sg g -- ainsuse omastav):",
        header_placement="middle",
    ),
    17: PhraseClassificationConfiguration(
        phrase_labels=["analüüsitav sõna"],
        class_labels={"sg n": "sg n", "sg g": "sg g", "sg p": "sg p"},
        header="Vali sõna morfoloogiline vorm (sg n - ainsuse nimetav, sg g - ainsuse omastav, sg p - ainsuse osastav):",
        header_placement="middle",
    ),
    19: PhraseClassificationConfiguration(
        phrase_labels=["analüüsitav sõna"],
        class_labels={"sg g": "sg g", "sg p": "sg p", "adt": "adt"},
        header="Vali sõna morfoloogiline vorm (sg g - ainsuse omastav, sg p - ainsuse osastav, adt - lühike sisseütlev):",
        header_placement="middle",
    ),
}

In [31]:
# Collect input files
input_files = []
input_dir = "./homonymous_word_forms/annotations/"
for fname in os.listdir(input_dir):
    if os.path.isdir(os.path.join(input_dir, fname)):
        for subfname in os.listdir(os.path.join(input_dir, fname)):
            if subfname.endswith(".json"):
                inflection_type = int(subfname.split("_")[2])  # infl_type_xx_1000_vx...
                input_files.append(
                    (
                        inflection_type,
                        "./homonymous_word_forms/annotations/" + fname + "/" + subfname,
                    )
                )

if not input_files:
    raise RuntimeError("No input files found!")

print(f"Found {len(input_files)} input files.")

Found 8 input files.


In [32]:
overall_data = []
# Extract data from input files
for infl_type, input_file in input_files:
    print(f"Processing file: {input_file} (inflection type {infl_type})")
    num = input_file.split("/")[3]
    annotation_conf = annotation_confs[infl_type]
    with open(input_file, "r", encoding="utf-8") as f:
        raw = f.read()
    task = PhraseClassificationTask(
        annotation_conf,
        input_layer="morph",
        output_layer="morph",
        label_attribute="label",
    )
    classified_sentences = task.import_data(raw)

    # Create dataframe from classified sentences
    data = []
    for sentence in classified_sentences:
        for annotation in sentence.morph:
            if (
                "class_label" in sentence.meta
                and sentence.meta["class_label"] is not None
            ):
                data.append(
                    {
                        "num": num,
                        "inflection_type": infl_type,
                        "sentence": sentence.text,
                        "word": annotation.text,
                        "word_span": (annotation.start, annotation.end),
                        "label": sentence.meta["class_label"],
                    }
                )
    df = pd.DataFrame(data)
    output_csv = (
        f"./homonymous_word_forms/processed/homonyms_infltype_{num}_{infl_type}.csv"
    )
    df.to_csv(output_csv, index=False)
    print(f"Saved processed data to {output_csv}")
    overall_data.extend(data)

# Create overall dataframe
overall_df = pd.DataFrame(overall_data)
overall_output_csv = "./homonymous_word_forms/processed/homonyms_overall.csv"
overall_df.to_csv(overall_output_csv, index=False)
print(f"Saved overall processed data to {overall_output_csv}")

Processing file: ./homonymous_word_forms/annotations/1/infl_type_01_1000_v1_project-1-at-2024-11-23-00-10-742338cd.json (inflection type 1)
Saved processed data to ./homonymous_word_forms/processed/homonyms_infltype_1_1.csv
Processing file: ./homonymous_word_forms/annotations/1/infl_type_16_1000_v1_project-2-at-2024-11-21-19-27-9e8ae0c2.json (inflection type 16)
Saved processed data to ./homonymous_word_forms/processed/homonyms_infltype_1_16.csv
Processing file: ./homonymous_word_forms/annotations/1/infl_type_17_1000_v1.json (inflection type 17)
Saved processed data to ./homonymous_word_forms/processed/homonyms_infltype_1_17.csv
Processing file: ./homonymous_word_forms/annotations/1/infl_type_19_1000_v1_project-6-at-2025-11-15-14-13-42753676.json (inflection type 19)
Saved processed data to ./homonymous_word_forms/processed/homonyms_infltype_1_19.csv
Processing file: ./homonymous_word_forms/annotations/2/infl_type_01_1000_v2_project-5-at-2024-12-11-21-53-280753b4.json (inflection type 

<a id='hindamine'></a>


### Evaluation (BertMorphTagger vs Vabamorf)


In [7]:
from estnltk.default_resolver import make_resolver

# Load BertMorphTagger model
bmt_model_name = "./NER_mudel_v2/"
bmt_model = BertMorphTagger(model_location=bmt_model_name)

# Load default resolver
resolver = make_resolver()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 863.93it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              


In [2]:
overall_df = pd.read_csv("./homonymous_word_forms/processed/homonyms_overall.csv")

In [14]:
overall_df.groupby(["inflection_type", "label"]).size().reset_index().sort_values(
    by="inflection_type"
)

,inflection_type,label,0
0,1,['sg g'],1232
1,1,['sg n'],764
2,16,['sg g'],1080
3,16,['sg n'],890
4,17,['sg g'],524
5,17,['sg n'],791
6,17,['sg p'],609
7,19,['adt'],94
8,19,['sg g'],1621
9,19,['sg p'],281


In [8]:
overall_df.head()

,num,inflection_type,sentence,word,word_span,label
0,1,1,"Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri võitja James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.",võitja,"(74, 80)",[sg n]
1,1,1,"Normi-aktiveerimise teooria (Schwartz, 1970) on algselt mõeldud moraalse otsustamisprotsessi analüüsimiseks abistava käitumise näitel.",teooria,"(20, 27)",[sg n]
2,1,1,"""Ehk oleks mõttekas ka mõni selleteemaline hoiatav kampaania korraldada,"" lisab punase autoga preili.",kampaania,"(51, 60)",[sg n]
3,1,1,"""Minu otsus oli õige ning teeksin kõik sama moodi, kui saaksin uuesti teha,"" kommenteerib kolm aastat tagasi eriliste teenete eest Eesti passi saanud Primakov.",õige,"(16, 20)",[sg n]
4,1,1,"Itaalia president ütles Venemaa riigipea auks korraldatud suurejoonelisel banketil, et kahe riigi ühisavaldus Iraagi kohta oli kahe riigipea ""suur tarkuseavaldus"".",Itaalia,"(0, 7)",[sg g]


In [18]:
outer = tqdm(
    overall_df.iterrows(),
    total=len(overall_df),
    desc="Evaluating BertMorphTagger vs Vabamorf",
)


# Evaluate BertMorphTagger vs Vabamorf
results = []
for index, row in outer:
    sentence_text = row["sentence"]
    num = row["num"]
    inflection_type = row["inflection_type"]
    word_to_analyze = row["word"]
    word_span = row["word_span"]
    true_label = row["label"][0]

    # Create EstNLTK Text object
    text = estnltk.Text(sentence_text)
    text.tag_layer("sentences")

    # Apply BertMorphTagger
    bmt_model.tag(text)

    # Get BertMorphTagger prediction for the word
    # in the new bert_morph_tagging layer
    bmt_prediction = None
    for annotation in text.bert_morph_tagging:
        if annotation.start == word_span[0] and annotation.end == word_span[1]:
            bmt_prediction = annotation.form[0]  # Get the first analysis
            # If the form is still a list, take the first element
            if isinstance(bmt_prediction, list):
                bmt_prediction = bmt_prediction[0]

    # Apply Vabamorf
    text.tag_layer(resolver=resolver)

    # Get Vabamorf prediction for the word
    vabamorf_prediction = None
    for annotation in text.morph_analysis:
        if annotation.start == word_span[0] and annotation.end == word_span[1]:
            vabamorf_prediction = annotation.form[0]  # Get the first analysis
            # If the form is still a list, take the first element
            if isinstance(vabamorf_prediction, list):
                vabamorf_prediction = vabamorf_prediction[0]

    results.append(
        {
            "num": num,
            "inflection_type": inflection_type,
            "sentence": sentence_text,
            "word": word_to_analyze,
            "true_label": true_label,
            "bmt_prediction": bmt_prediction,
            "vabamorf_prediction": vabamorf_prediction,
        }
    )
    outer.refresh()

# Create results dataframe
results_df = pd.DataFrame(results)
results_output_csv = "./homonymous_word_forms/processed/homonyms_evaluation_results.csv"
results_df.to_csv(results_output_csv, index=False)

Evaluating BertMorphTagger vs Vabamorf:  18%|█▊        | 1454/7886 [01:52<06:56, 15.44it/s]e:\Git_projects\EstNLTK\morf_yhestaja\bert_tokens_to_words_rewriter.py:154: UserWarning: (!) No matching words span for bert token Span(' ', [{'bert_tokens': '▁', 'form': 'sg g', 'partofspeech': 'S', 'probability': 0.63123}]).
  warnings.warn(f"(!) No matching {words_layer.name} span for bert token {bert_tokens_layer[i]}.")
Evaluating BertMorphTagger vs Vabamorf:  35%|███▌      | 2769/7886 [03:24<06:00, 14.19it/s]e:\Git_projects\EstNLTK\morf_yhestaja\bert_tokens_to_words_rewriter.py:154: UserWarning: (!) No matching words span for bert token Span(' ', [{'bert_tokens': '▁', 'form': '?', 'partofspeech': '', 'probability': 0.94938}]).
  warnings.warn(f"(!) No matching {words_layer.name} span for bert token {bert_tokens_layer[i]}.")
Evaluating BertMorphTagger vs Vabamorf:  43%|████▎     | 3403/7886 [04:12<04:51, 15.37it/s]e:\Git_projects\EstNLTK\morf_yhestaja\bert_tokens_to_words_rewriter.py:154: Use

<a id='tulemused'></a>


### Results


In [2]:
# Load results dataframe
results_df = pd.read_csv(
    "./homonymous_word_forms/processed/homonyms_evaluation_results.csv", index_col=False
)

# Fill NaN predictions with "no_prediction"
results_df["bmt_prediction"] = results_df["bmt_prediction"].fillna("no_prediction")
results_df["vabamorf_prediction"] = results_df["vabamorf_prediction"].fillna(
    "no_prediction"
)

# Generate classification reports for both models
bmt_results_cr = sklearn.metrics.classification_report(
    y_true=results_df["true_label"],
    y_pred=results_df["bmt_prediction"],
    zero_division=0,
)

vabamorf_results_cr = sklearn.metrics.classification_report(
    y_true=results_df["true_label"],
    y_pred=results_df["vabamorf_prediction"],
    zero_division=0,
)

In [26]:
# Print classification reports
print("Overall Evaluation Results:")
print("BertMorphTagger Classification Report:")
print(bmt_results_cr)
print("Vabamorf Classification Report:")
print(vabamorf_results_cr)

Overall Evaluation Results:
BertMorphTagger Classification Report:
               precision    recall  f1-score   support

            ?       0.00      0.00      0.00         0
          adt       0.66      0.35      0.46        94
           da       0.00      0.00      0.00         0
           ge       0.00      0.00      0.00         0
           ma       0.00      0.00      0.00         0
           me       0.00      0.00      0.00         0
        neg o       0.00      0.00      0.00         0
no_prediction       0.00      0.00      0.00         0
            o       0.00      0.00      0.00         0
         pl g       0.00      0.00      0.00         0
         pl n       0.00      0.00      0.00         0
         pl p       0.00      0.00      0.00         0
            s       0.00      0.00      0.00         0
       sg all       0.00      0.00      0.00         0
        sg el       0.00      0.00      0.00         0
        sg es       0.00      0.00      0.00        

In [27]:
# Generate classification reports per inflection type
for infl_type in results_df["inflection_type"].unique():
    # Generate classification report for the current inflection type
    bmt_results_cr_infl = sklearn.metrics.classification_report(
        y_true=results_df[results_df["inflection_type"] == infl_type]["true_label"],
        y_pred=results_df[results_df["inflection_type"] == infl_type]["bmt_prediction"],
        zero_division=0,
    )

    vabamorf_results_cr_infl = sklearn.metrics.classification_report(
        y_true=results_df[results_df["inflection_type"] == infl_type]["true_label"],
        y_pred=results_df[results_df["inflection_type"] == infl_type][
            "vabamorf_prediction"
        ],
        zero_division=0,
    )

    print(f"{'=' * 8}Inflection Type {infl_type}{'=' * 8}")
    print("BertMorphTagger Classification Report:")
    print(bmt_results_cr_infl)
    print("Vabamorf Classification Report:")
    print(vabamorf_results_cr_infl)

========Inflection Type 1========
BertMorphTagger Classification Report:
               precision    recall  f1-score   support

            ?       0.00      0.00      0.00         0
          adt       0.00      0.00      0.00         0
           ge       0.00      0.00      0.00         0
no_prediction       0.00      0.00      0.00         0
         pl g       0.00      0.00      0.00         0
         pl p       0.00      0.00      0.00         0
       sg all       0.00      0.00      0.00         0
        sg es       0.00      0.00      0.00         0
         sg g       0.91      0.90      0.90      1232
        sg in       0.00      0.00      0.00         0
       sg kom       0.00      0.00      0.00         0
         sg n       0.87      0.83      0.85       764
         sg p       0.00      0.00      0.00         0

     accuracy                           0.87      1996
    macro avg       0.14      0.13      0.13      1996
 weighted avg       0.90      0.87      0.88 

### Baseline models (random choice and most frequent choice)


### Label distribution


In [15]:
# View label distribution in the dataset

# Count occurrences of each label in the entire dataset
label_distribution_overall = (
    results_df.groupby("true_label").size().reset_index(name="count")
)

# Calculate percentages
label_distribution_overall["percentage"] = label_distribution_overall["count"].apply(
    lambda x: round(x / label_distribution_overall["count"].sum() * 100, 3)
)
print("=" * 5, "Overall Label Distribution", "=" * 5)
print(label_distribution_overall, "\n")


# View label distribution per inflection type and add percentage columns
print("=" * 5, "Label Distribution per Inflection Type", "=" * 5)

# Count occurrences of each label per inflection type
label_distribution = (
    results_df.groupby(["inflection_type", "true_label"])
    .size()
    .reset_index(name="count")
)

# Calculate percentages
label_distribution["percentage"] = label_distribution.groupby("inflection_type")[
    "count"
].transform(lambda x: round(x / x.sum() * 100, 3))

# Separate dataframes per inflection type
label_distribution_dfs = {}
for infl_type in label_distribution["inflection_type"].unique():
    label_distribution_dfs[infl_type] = label_distribution[
        label_distribution["inflection_type"] == infl_type
    ].reset_index(drop=True)
    print(f"Inflection Type {infl_type} Label Distribution:")
    print(label_distribution_dfs[infl_type])

===== Overall Label Distribution =====
  true_label  count  percentage
0        adt     94       1.192
1       sg g   4457      56.518
2       sg n   2445      31.004
3       sg p    890      11.286 

===== Label Distribution per Inflection Type =====
Inflection Type 1 Label Distribution:
   inflection_type true_label  count  percentage
0                1       sg g   1232      61.723
1                1       sg n    764      38.277
Inflection Type 16 Label Distribution:
   inflection_type true_label  count  percentage
0               16       sg g   1080      54.822
1               16       sg n    890      45.178
Inflection Type 17 Label Distribution:
   inflection_type true_label  count  percentage
0               17       sg g    524      27.235
1               17       sg n    791      41.112
2               17       sg p    609      31.653
Inflection Type 19 Label Distribution:
   inflection_type true_label  count  percentage
0               19        adt     94       4.709
1    

#### Overall homonym dataset


In [29]:
seed = 42069360
# Create a most frequent class baseline
most_frequent_baseline = sklearn.dummy.DummyClassifier(
    strategy="most_frequent", random_state=seed
)
# Create a random choice baseline
random_choice_baseline = sklearn.dummy.DummyClassifier(
    strategy="uniform", random_state=seed
)

In [30]:
# Fit and evaluate baselines
for baseline, name in [
    (most_frequent_baseline, "Most Frequent Class Baseline"),
    (random_choice_baseline, "Random Choice Baseline"),
]:
    baseline.fit(
        results_df["true_label"],
        results_df["true_label"],
    )
    baseline_predictions = baseline.predict(
        results_df["true_label"],
    )
    baseline_cr = sklearn.metrics.classification_report(
        y_true=results_df["true_label"],
        y_pred=baseline_predictions,
        zero_division=0,
    )
    print(f"{name} Classification Report:")
    print(baseline_cr)

Most Frequent Class Baseline Classification Report:
              precision    recall  f1-score   support

         adt       0.00      0.00      0.00        94
        sg g       0.57      1.00      0.72      4457
        sg n       0.00      0.00      0.00      2445
        sg p       0.00      0.00      0.00       890

    accuracy                           0.57      7886
   macro avg       0.14      0.25      0.18      7886
weighted avg       0.32      0.57      0.41      7886

Random Choice Baseline Classification Report:
              precision    recall  f1-score   support

         adt       0.01      0.27      0.02        94
        sg g       0.55      0.25      0.34      4457
        sg n       0.31      0.25      0.28      2445
        sg p       0.11      0.25      0.16       890

    accuracy                           0.25      7886
   macro avg       0.25      0.25      0.20      7886
weighted avg       0.42      0.25      0.30      7886



#### Per inflection type homonym dataset


In [31]:
seed = 42069360
# Create a most frequent class baseline
most_frequent_baseline = sklearn.dummy.DummyClassifier(
    strategy="most_frequent", random_state=seed
)
# Create a random choice baseline
random_choice_baseline = sklearn.dummy.DummyClassifier(
    strategy="uniform", random_state=seed
)

In [34]:
# Fit and evaluate baselines
for baseline, name in [
    (most_frequent_baseline, "Most Frequent Class Baseline"),
    (random_choice_baseline, "Random Choice Baseline"),
]:
    for inflection_type in results_df["inflection_type"].unique():
        baseline.fit(
            results_df[results_df["inflection_type"] == inflection_type]["true_label"],
            results_df[results_df["inflection_type"] == inflection_type]["true_label"],
        )
        baseline_predictions = baseline.predict(
            results_df[results_df["inflection_type"] == inflection_type]["true_label"],
        )
        baseline_cr = sklearn.metrics.classification_report(
            y_true=results_df[results_df["inflection_type"] == inflection_type][
                "true_label"
            ],
            y_pred=baseline_predictions,
            zero_division=0,
        )
        print(f"{name} Classification Report for Inflection Type {inflection_type}:")
        print(baseline_cr)

Most Frequent Class Baseline Classification Report for Inflection Type 1:
              precision    recall  f1-score   support

        sg g       0.62      1.00      0.76      1232
        sg n       0.00      0.00      0.00       764

    accuracy                           0.62      1996
   macro avg       0.31      0.50      0.38      1996
weighted avg       0.38      0.62      0.47      1996

Most Frequent Class Baseline Classification Report for Inflection Type 16:
              precision    recall  f1-score   support

        sg g       0.55      1.00      0.71      1080
        sg n       0.00      0.00      0.00       890

    accuracy                           0.55      1970
   macro avg       0.27      0.50      0.35      1970
weighted avg       0.30      0.55      0.39      1970

Most Frequent Class Baseline Classification Report for Inflection Type 17:
              precision    recall  f1-score   support

        sg g       0.00      0.00      0.00       524
        sg n 

## END
